# LLM-Firewall: fine-tune Layer C (transformer) on a free Colab GPU

This notebook runs the **same** training script as the repo (`training/train_transformer.py`), just on a GPU.

1. `Runtime -> Change runtime type -> T4 GPU`
2. `Runtime -> Run all` (about 5-10 minutes)
3. Download `transformer_model.zip`, unzip it into `models/transformer/` in your local clone.
4. Locally: `python -m eval.run_eval --transformer-dir models/transformer` to get real Layer C numbers.

The model is ~260 MB, which is why it is not committed to git (the repo keeps files under 50 MB).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: training will be slow'

In [ ]:
!git clone --depth 1 https://github.com/ibragot/llm-firewall.git
%cd llm-firewall
# Colab already has torch with CUDA; add the rest.
!pip install -q -r requirements-ml.txt "transformers>=4.40" "tokenizers>=0.15" "accelerate>=0.28"

## Train

* `distilbert-base-uncased`: 66M parameters, English-only vocabulary, fast.
* Swap to `--model microsoft/deberta-v3-small` (needs `pip install sentencepiece`) for a stronger but slower model,
  or `--model distilbert-base-multilingual-cased` if multilingual attacks matter most to you.

The script prints validation ROC-AUC / average precision / F1 after every epoch and keeps the best epoch.

In [ ]:
!python -m training.train_transformer --model distilbert-base-uncased --epochs 3 --batch-size 32 --lr 3e-5

In [ ]:
!cat models/transformer/training_metrics.json

## Evaluate all layers on the held-out red-team suite and public test splits

In [ ]:
!python -m eval.run_eval --transformer-dir models/transformer

In [ ]:
from IPython.display import Image, display
for name in ["roc", "detection_by_category", "threshold_sweep", "latency"]:
    display(Image(f"eval/results/{name}.png"))

## Package the model for download

In [ ]:
!cd models && zip -qr ../transformer_model.zip transformer && cd .. && ls -lh transformer_model.zip
!zip -qr eval_results_with_transformer.zip eval/results
from google.colab import files
files.download("transformer_model.zip")
files.download("eval_results_with_transformer.zip")